In [9]:
import pandas as pd
import seaborn as sns
import numpy as np

In [4]:
transaction_pd = pd.read_csv('C:/Users/Playdata/Downloads/h-and-m-personalized-fashion-recommendations/transactions_train.csv')
customer_df = pd.read_csv('C:/Users/Playdata/Downloads/h-and-m-personalized-fashion-recommendations/customers.csv')
art_df = pd.read_csv('C:/Users/Playdata/Downloads/h-and-m-personalized-fashion-recommendations/articles.csv')

In [5]:
print(len(transaction_pd))
print(len(customer_df))
print(len(art_df))

31788324
1371980
105542


In [6]:
art_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 105542 entries, 0 to 105541
Data columns (total 25 columns):
 #   Column                        Non-Null Count   Dtype 
---  ------                        --------------   ----- 
 0   article_id                    105542 non-null  int64 
 1   product_code                  105542 non-null  int64 
 2   prod_name                     105542 non-null  object
 3   product_type_no               105542 non-null  int64 
 4   product_type_name             105542 non-null  object
 5   product_group_name            105542 non-null  object
 6   graphical_appearance_no       105542 non-null  int64 
 7   graphical_appearance_name     105542 non-null  object
 8   colour_group_code             105542 non-null  int64 
 9   colour_group_name             105542 non-null  object
 10  perceived_colour_value_id     105542 non-null  int64 
 11  perceived_colour_value_name   105542 non-null  object
 12  perceived_colour_master_id    105542 non-null  int64 
 13 

In [7]:
# transaction & customer 중복 key 컬럼 확인
print(set(transaction_pd.columns)&set(customer_df.columns))

# customer & transaction merge (key : customer id)
cust_tran_df = customer_df.merge(transaction_pd, how='inner', on = ['customer_id'])

# cust_tran_df.isna().sum()

{'customer_id'}


In [10]:
set(art_df.columns)&set(cust_tran_df.columns)
total_df = cust_tran_df.merge(art_df, how='inner', on =['article_id'])

MemoryError: Unable to allocate 243. MiB for an array with shape (31788324,) and data type int64

In [ ]:
# 패션 뉴스, 온라인 마케팅 등록 여부
total_df['FN'] = total_df['FN'].fillna(0)
total_df['Active'] = total_df['Active'].fillna(0)

In [ ]:
# FN, ACtive 0 채우기 확인
total_df.isna().sum()

customer_id                          0
FN                                   0
Active                               0
club_member_status               62165
fashion_news_frequency          141713
age                             140258
postal_code                          0
t_dat                                0
article_id                           0
price                                0
sales_channel_id                     0
product_code                         0
prod_name                            0
product_type_no                      0
product_type_name                    0
product_group_name                   0
graphical_appearance_no              0
graphical_appearance_name            0
colour_group_code                    0
colour_group_name                    0
perceived_colour_value_id            0
perceived_colour_value_name          0
perceived_colour_master_id           0
perceived_colour_master_name         0
department_no                        0
department_name          

### club_member_status 
- H&M 클럽 멤버십은 회원이 계정을 생성해 가입한 뒤, 구매·리뷰·기타 활동을 통해 포인트를 적립하고 이를 리워드로 교환하는 로열티 프로그램
- ACTIVE: 현재 클럽 멤버십이 활성화된 회원
- PRE-CREATE: 계정/멤버십이 사전 생성되었으나 완전 활성화 전 상태
- LEFT CLUB: 클럽을 탈퇴한 회원 (브랜드 완전이탈이 확정이 아님, 조금 애매함)
- NaN: 멤버십 상태 정보가 없는 결측값

**H&M club_member_status NaN

오프라인 구매만 한 고객,
멤버십 상태 수집 전 가입자,
데이터 병합 과정에서 상태 누락,
멤버십과 무관한 일반 고객

In [ ]:
total_df['club_member_status'].unique()

array(['ACTIVE', nan, 'PRE-CREATE', 'LEFT CLUB'], dtype=object)

In [ ]:
# club_member_status na값 삭제
# 62,000(NA) / 30,000,000 ≈ 0.21%
total_df['club_member_status'] = total_df['club_member_status'].replace('nan',np.nan)
total_df = total_df.dropna(subset=['club_member_status'])

In [ ]:
total_df.isna().sum()

customer_id                          0
FN                                   0
Active                               0
club_member_status                   0
fashion_news_frequency          129103
age                             126598
postal_code                          0
t_dat                                0
article_id                           0
price                                0
sales_channel_id                     0
product_code                         0
prod_name                            0
product_type_no                      0
product_type_name                    0
product_group_name                   0
graphical_appearance_no              0
graphical_appearance_name            0
colour_group_code                    0
colour_group_name                    0
perceived_colour_value_id            0
perceived_colour_value_name          0
perceived_colour_master_id           0
perceived_colour_master_name         0
department_no                        0
department_name          

#### 패션 뉴스레터 수신 빈도 
- array(['NONE', 'Regularly', nan, 'Monthly'], dtype=object)

| 값         | 의미              |
| --------- | --------------- |
| NONE      | 수신 안 함 /뉴스레터 거부자와 정보 없음 고객을 같은 그룹|
| Monthly   | 월 1회 수신         |
| Regularly | 정기 수신 (월 1회 이상) |
| NaN       | 정보 없음           |


In [ ]:
total_df['fashion_news_frequency'] = total_df['fashion_news_frequency'].fillna('UNKNOWN')
print(total_df['fashion_news_frequency'].unique())
total_df.isna().sum()

['NONE' 'Regularly' 'UNKNOWN' 'Monthly']


customer_id                          0
FN                                   0
Active                               0
club_member_status                   0
fashion_news_frequency               0
age                             126598
postal_code                          0
t_dat                                0
article_id                           0
price                                0
sales_channel_id                     0
product_code                         0
prod_name                            0
product_type_no                      0
product_type_name                    0
product_group_name                   0
graphical_appearance_no              0
graphical_appearance_name            0
colour_group_code                    0
colour_group_name                    0
perceived_colour_value_id            0
perceived_colour_value_name          0
perceived_colour_master_id           0
perceived_colour_master_name         0
department_no                        0
department_name          

### 나이 (결측치 알수없음으로 채움)
- min(10대), max(90대) - 이상치인지 확인 필요
 (16.0, 99.0)


In [ ]:
min(total_df['age']), max(total_df['age'])

(16.0, 99.0)

In [ ]:
total_df[total_df['age']>=90].value_counts() # 1393건

customer_id                                                       FN   Active  club_member_status  fashion_news_frequency  age   postal_code                                                       t_dat       article_id  price     sales_channel_id  product_code  prod_name            product_type_no  product_type_name  product_group_name  graphical_appearance_no  graphical_appearance_name  colour_group_code  colour_group_name  perceived_colour_value_id  perceived_colour_value_name  perceived_colour_master_id  perceived_colour_master_name  department_no  department_name     index_code  index_name          index_group_no  index_group_name  section_no  section_name                    garment_group_no  garment_group_name  detail_desc                                                                                                                                                                                    
bc870dbac4a150f6e5cf8e1d71e59d77d3da3f171dae2433b7d93dd3341e3b07  0.0  0.0     ACTI

In [ ]:
bins = [0, 19, 29, 39, 49, 59, 69, 100]
labels = ['10대 미만', '10대', '20대', '30대', '40대', '50대', '60대 이상']
total_df['age_cut'] = pd.cut(total_df['age'], bins=bins, labels=labels, right=True).astype('str')
type(total_df['age_cut'])
total_df['age_cut'] = total_df['age_cut'].str.replace('nan', 'UNKNOWN')


In [ ]:
total_df['age_cut'].unique()

array(['30대', '10대', '40대', '20대', '60대 이상', '50대', '10대 미만', 'UNKNOWN'],
      dtype=object)

In [ ]:
total_df['age_cut'].value_counts()

age_cut
10대        13039351
20대         6416888
40대         5130097
30대         4901185
50대         1201949
10대 미만       690192
60대 이상       219899
UNKNOWN      126598
Name: count, dtype: int64

In [ ]:
total_df.columns

Index(['customer_id', 'FN', 'Active', 'club_member_status',
       'fashion_news_frequency', 'age', 'postal_code', 't_dat', 'article_id',
       'price', 'sales_channel_id', 'product_code', 'prod_name',
       'product_type_no', 'product_type_name', 'product_group_name',
       'graphical_appearance_no', 'graphical_appearance_name',
       'colour_group_code', 'colour_group_name', 'perceived_colour_value_id',
       'perceived_colour_value_name', 'perceived_colour_master_id',
       'perceived_colour_master_name', 'department_no', 'department_name',
       'index_code', 'index_name', 'index_group_no', 'index_group_name',
       'section_no', 'section_name', 'garment_group_no', 'garment_group_name',
       'detail_desc', 'age_cut'],
      dtype='object')

In [ ]:
total_df.head(5)

,customer_id,FN,Active,club_member_status,fashion_news_frequency,age,postal_code,t_dat,article_id,price,...,index_code,index_name,index_group_no,index_group_name,section_no,section_name,garment_group_no,garment_group_name,detail_desc,age_cut
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,0.0,0.0,ACTIVE,NONE,49.0,52043ee2162cf5aa7ee79974281641c6f11a68d276429a...,2018-12-27,625548001,0.044051,...,I,Children Sizes 134-170,4,Baby/Children,45,Kids Outerwear,1007,Outdoor,"Padded jacket with a detachable hood, stand-up...",30대
1,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,0.0,0.0,ACTIVE,NONE,49.0,52043ee2162cf5aa7ee79974281641c6f11a68d276429a...,2018-12-27,176209023,0.035576,...,F,Menswear,3,Menswear,31,Mens Outerwear,1007,Outdoor,"Short, padded jacket with a jersey-lined hood ...",30대
2,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,0.0,0.0,ACTIVE,NONE,49.0,52043ee2162cf5aa7ee79974281641c6f11a68d276429a...,2018-12-27,627759010,0.030492,...,H,Children Sizes 92-140,4,Baby/Children,45,Kids Outerwear,1007,Outdoor,"Padded parka in woven fabric with a soft, brus...",30대
3,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,0.0,0.0,ACTIVE,NONE,49.0,52043ee2162cf5aa7ee79974281641c6f11a68d276429a...,2019-05-02,697138006,0.010153,...,H,Children Sizes 92-140,4,Baby/Children,76,Kids Girl,1005,Jersey Fancy,Playsuit in cotton jersey with butterfly sleev...,30대
4,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,0.0,0.0,ACTIVE,NONE,49.0,52043ee2162cf5aa7ee79974281641c6f11a68d276429a...,2019-05-25,568601006,0.050831,...,A,Ladieswear,1,Ladieswear,11,Womens Tailoring,1008,Dressed,Fitted jacket in woven fabric with notch lapel...,30대


### 1차 컬럼 분류

In [ ]:
total_df = total_df[['customer_id', 'FN', 'Active', 'club_member_status',
       'fashion_news_frequency', 'age', 't_dat', 'price','product_type_name', 'product_group_name',
       'graphical_appearance_name','colour_group_name','perceived_colour_value_name','perceived_colour_master_name', 'department_name',
        'index_name', 'index_group_name','section_name','garment_group_name','age_cut']]

In [ ]:
EDA_columns = total_df

In [ ]:
EDA_columns.head(5)

NameError: name 'EDA_columns' is not defined

### price 값 수치 의미 (기업 미공개- 정규화된 데이터)

| 값 예시                          | 해석                      |                 |
| -------------------------- | ----------------------- | --------------- |
| `0.009`                    | 원본에서 매우 낮은 가격 → 스케일링 결과 |                 |
| `0.0278` (dataset average) | 스케일링된 평균 거래 가격          | ([ARAMDAUN][1]) |
| `0.5915` (max)             | 스케일된 가장 비싼 상품           | ([ARAMDAUN][1]) |

[1]: https://ars420.tistory.com/42?utm_source=chatgpt.com "[Kaggle] H&M Personalized Fashion Recommendations"


In [ ]:
def get_season(month):
    if month in [3,4,5,6,7,8]:
        return "SS"
    else:
        return "FW"

In [ ]:
# total_df.to_csv('C:/Users/Playdata/Downloads/total_df.csv')

In [ ]:
# def get_season(date):
#     if date.month in [3,4,5,6,7,8]:
#         return "SS"
#     else:
#         return "FW"

# df["season"] = df["first_purchase_date"].apply(get_season)

# # 신규 고객 churn 계산
# df["days_since_first_purchase"] = (df["last_purchase_date"] - df["first_purchase_date"]).dt.days
# df["churn"] = 0
# df.loc[(df["season"]=="SS") & (df["days_since_first_purchase"]>60), "churn"]=1
# df.loc[(df["season"]=="FW") & (df["days_since_first_purchase"]>90), "churn"]=1